# 🚀 Google Colab Setup Instructions

**IMPORTANT: Enable GPU Runtime**
1. Go to `Runtime` → `Change runtime type`
2. Select `GPU` as Hardware accelerator
3. Choose T4, V100, or A100 (if available)
4. Click `Save`

**Note:** This notebook requires GPU for efficient training. Without GPU, training will be extremely slow.

## 📁 Optional: Mount Google Drive (for data storage)

In [ ]:
# Uncomment the lines below if you want to mount Google Drive
# This is useful for accessing datasets or saving models to Drive

# from google.colab import drive
# drive.mount('/content/drive')
# print("✓ Google Drive mounted at /content/drive")

# Example: If your data is in Google Drive
# TRAIN_FILE = "/content/drive/MyDrive/your-folder/data/train.csv"
# TEST_FILE = "/content/drive/MyDrive/your-folder/data/phase_1_test.csv"

# QWEN Fine-Tuning for 5G Network Troubleshooting

This notebook fine-tunes QWEN 1.5B models on telecom troubleshooting data using LoRA for efficient training on Google Colab with GPU support.

## Dataset Overview
- **Size**: 112,806 training samples
- **Format**: CSV with ID, question, answer columns
- **Task**: Multi-choice classification (8 root cause options)

- Adjust data paths as needed for your Colab environment

## Requirements- Google Colab with GPU runtime (T4, V100, or A100 recommended)

In [1]:
# ============================================================================
# 1. ENVIRONMENT SETUP & IMPORTS
# ============================================================================

import torch
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check PyTorch and device setup
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = "cuda"
else:
    print("WARNING: GPU not available, using CPU (training will be very slow)")
    device = "cpu"


print(f"Using device: {device}")print()

ENVIRONMENT CHECK
PyTorch version: 2.9.1
MPS available: True
MPS built: True
Using device: mps
MPS metal performance shaders enabled



In [2]:
# Install required packages for Colab
import subprocess
import sys

packages = [
    "transformers>=4.36.0",
    "peft>=0.7.0",  # For LoRA
    "datasets>=2.14.0",
    "accelerate>=0.24.0",
    "bitsandbytes>=0.41.0",  # For 4-bit/8-bit quantization on GPU
    "scipy",
]

print("Installing required packages...")
for package in packages:
    print(f"  Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All packages installed")

# Import after installation
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
import gc

    print("✓ GPU cache cleared")

print("✓ All imports successful")    torch.cuda.empty_cache()

if torch.cuda.is_available():
# Clear GPU cache

Installing required packages...
✓ All packages installed
✓ All packages installed
✓ All imports successful
✓ All imports successful


## 2. Data Loading & Preprocessing

In [3]:
TRAIN_FILE = "data/train.csv"
TEST_FILE = "data/phase_1_test.csv"


df_train = pd.read_csv(TRAIN_FILE)
display(df_train.head())
print(f"✓ Loaded {len(df_train)} training samples")
print(f"Columns: {df_train.columns.tolist()}")
print(f"\nFirst sample:")
print(f"Question preview: {df_train['question'].iloc[0][:200]}...")
print(f"Answer: {df_train['answer'].iloc[0]}")
print(f"\nAnswer distribution:")
print(df_train['answer'].value_counts())

,ID,question,answer
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test use...,C2
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test use...,C1
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test use...,C2
3,ID_D6C9N2X295,Analyze the 5G wireless network drive-test use...,C2
4,ID_8JC15PNP3Q,Analyze the 5G wireless network drive-test use...,C5


✓ Loaded 2400 training samples
Columns: ['ID', 'question', 'answer']

First sample:
Question preview: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 pote...
Answer: C2

Answer distribution:
answer
C5    352
C7    349
C3    330
C2    320
C4    283
C8    277
C1    264
C6    225
Name: count, dtype: int64


In [4]:
import re
import json
import math
import pandas as pd
from statistics import mean

CAUSE_TO_NUM = {f"C{i}": i for i in range(1, 9)}

# -----------------------------
# NEW: sanitize question text
# -----------------------------
def sanitize_question_text(q: str) -> str:
    """
    Your CSV stores literal '\\n' characters. Convert only those to real newlines.
    Do NOT use unicode decoding, or you may corrupt '\\boxed' via '\\b'.
    """
    if q is None:
        return ""
    return q.replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

# -----------------------------
# Helpers: geometry + beam rules
# -----------------------------
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6
    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6
    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    if 6 <= k <= 11:
        return 12
    return 25

def electronic_tilt_deg(digital_tilt):
    # 255 => 6 degrees, otherwise value is degrees
    try:
        v = int(float(digital_tilt))
    except Exception:
        return 6.0
    return 6.0 if v == 255 else float(v)

# -----------------------------
# 1) Parse pipe tables from text
# -----------------------------
def parse_pipe_table(table_text: str):
    """Parse a pipe-delimited table block into list[dict]. Handles '-' as None."""
    if not table_text:
        return []

    lines = [ln.strip() for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if "|" in ln]
    if not lines:
        return []

    header = [h.strip() for h in lines[0].split("|")]
    rows = []
    for ln in lines[1:]:
        parts = [p.strip() for p in ln.split("|")]

        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        if len(parts) > len(header):
            parts = parts[:len(header)]

        rec = {}
        for k, v in zip(header, parts):
            rec[k] = None if v in ("", "-", "—") else v
        rows.append(rec)

    return rows

# -----------------------------
# 2) Normalize columns to stable keys
# -----------------------------
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",

    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",

    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

def to_float(x):
    try:
        return float(x)
    except Exception:
        return None

def to_int(x):
    try:
        return int(float(x))
    except Exception:
        return None

def normalize_rows(rows, mapping):
    out = []
    for r in rows:
        nr = {}
        for k, v in r.items():
            nk = mapping.get(k)
            if nk is None:
                continue
            nr[nk] = v

        # numeric casting
        for f in ["longitude", "latitude", "gps_speed_kmh", "ss_rsrp_dbm", "ss_sinr_db",
                  "throughput_mbps", "dl_rb_num", "mechanical_downtilt", "digital_tilt",
                  "height", "max_tx_power", "mechanical_azimuth", "digital_azimuth"]:
            if f in nr:
                nr[f] = to_float(nr[f])

        for f in ["pci", "serving_pci", "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            if f in nr:
                nr[f] = to_int(nr[f])

        out.append(nr)
    return out

# -----------------------------
# 3) Derived feature summary
# -----------------------------
def summarize_example(drive_rows, eng_rows):
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]

    tp_min = min(tps) if tps else None
    tp_mean = mean(tps) if tps else None
    tp_drop_ratio = (sum(1 for x in tps if x < 600) / len(tps)) if tps else None

    rb_mean = mean(rbs) if rbs else None
    rb_min = min(rbs) if rbs else None

    speed_max = max(speeds) if speeds else None

    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1

    neighbor_pcis = []
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            v = r.get(k)
            if v is not None:
                neighbor_pcis.append(v)

    mod30_hit = False
    if serving_pcis and neighbor_pcis:
        sp = serving_pcis[-1]
        mod30_hit = any((n % 30) == (sp % 30) for n in neighbor_pcis if n is not None)

    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}

    dists = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)
        if not cell:
            continue
        if r.get("latitude") is None or r.get("longitude") is None:
            continue
        if cell.get("latitude") is None or cell.get("longitude") is None:
            continue
        dists.append(haversine_m(r["latitude"], r["longitude"], cell["latitude"], cell["longitude"]))

    dist_median = sorted(dists)[len(dists) // 2] if dists else None
    dist_p95 = sorted(dists)[int(0.95 * (len(dists) - 1))] if dists else None

    tilt_total = None
    vbw = None
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])
        if cell:
            mech = cell.get("mechanical_downtilt") or 0.0
            digi = electronic_tilt_deg(cell.get("digital_tilt"))
            tilt_total = float(mech) + float(digi)
            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))

    return {
        "tp_min": tp_min, "tp_mean": tp_mean, "tp_drop_ratio": tp_drop_ratio,
        "rb_mean": rb_mean, "rb_min": rb_min,
        "speed_max": speed_max,
        "handover_count": handover_count,
        "pci_mod30_collision": mod30_hit,
        "dist_median_m": dist_median,
        "dist_p95_m": dist_p95,
        "serving_total_tilt_deg": tilt_total,
        "serving_vertical_beamwidth_deg": vbw,
    }

def format_user_prompt(summary, drive_rows, eng_rows):
    def fmt(v):
        if v is None:
            return "NA"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, int):
            return str(v)
        if isinstance(v, float):
            return f"{v:.2f}"
        return str(v)

    summary_block = "\n".join([
        f"- tp_min_mbps: {fmt(summary['tp_min'])}",
        f"- tp_mean_mbps: {fmt(summary['tp_mean'])}",
        f"- tp_drop_ratio(<600): {fmt(summary['tp_drop_ratio'])}",
        f"- rb_mean: {fmt(summary['rb_mean'])}",
        f"- rb_min: {fmt(summary['rb_min'])}",
        f"- speed_max_kmh: {fmt(summary['speed_max'])}",
        f"- handover_count: {fmt(summary['handover_count'])}",
        f"- pci_mod30_collision: {fmt(summary['pci_mod30_collision'])}",
        f"- dist_median_m: {fmt(summary['dist_median_m'])}",
        f"- dist_p95_m: {fmt(summary['dist_p95_m'])}",
        f"- serving_total_tilt_deg: {fmt(summary['serving_total_tilt_deg'])}",
        f"- serving_vertical_beamwidth_deg: {fmt(summary['serving_vertical_beamwidth_deg'])}",
    ])

    return f"""Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \\boxed{{}}.

C1: Downtilt too large, weak far coverage.
C2: Coverage distance exceeds 1km (overshoot).
C3: Neighbor cell provides higher throughput.
C4: Non-colocated co-frequency neighbors cause severe overlap.
C5: Frequent handovers degrade performance.
C6: Same PCI mod 30 causes interference.
C7: Speed exceeds 40km/h.
C8: Average scheduled RBs below 160.

Given:
- Digital tilt 255 => 6 degrees; otherwise value is degrees.
- Beam scenario => vertical beamwidth:
  Default/SCENARIO_1..5: 6 deg; SCENARIO_6..11: 12 deg; SCENARIO_12+: 25 deg.

Derived Feature Summary:
{summary_block}

Drive-test samples (canonical JSON):
{json.dumps(drive_rows, ensure_ascii=False)}

Engineering parameters (canonical JSON):
{json.dumps(eng_rows, ensure_ascii=False)}
"""

# -----------------------------
# 4) Row preprocessing: question text -> messages
# -----------------------------
def preprocess_row(row):
    # IMPORTANT FIX: sanitize literal \\n into real newlines
    q = sanitize_question_text(row["question"])
    label = str(row["answer"]).strip().upper()

    drive_match = re.search(r"User plane drive test data as follows[:：]\s*", q)
    eng_match = re.search(r"Engeneering parameters data as follows[:：]\s*", q)

    if not drive_match or not eng_match or eng_match.start() <= drive_match.end():
        return None

    drive_text = q[drive_match.end():eng_match.start()].strip()
    eng_text = q[eng_match.end():].strip()

    drive_raw = parse_pipe_table(drive_text)
    eng_raw = parse_pipe_table(eng_text)

    drive_rows = normalize_rows(drive_raw, DRIVE_MAP)
    eng_rows = normalize_rows(eng_raw, ENG_MAP)

    summary = summarize_example(drive_rows, eng_rows)
    user_content = format_user_prompt(summary, drive_rows, eng_rows)

    y = CAUSE_TO_NUM.get(label)
    if y is None:
        return None

    # Keep the assistant target stable and minimal
    assistant_content = f"Final answer: \\\\boxed{{{y}}}"

    return {
        "id": row["ID"],
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a senior 5G RAN optimization engineer. "
                    "Infer the most likely root cause (C1–C8) and end with \\boxed{n}."
                ),
            },
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ],
    }

# -----------------------------
# 5) Apply to df_train and export JSONL
# -----------------------------
records = []
for r in df_train.to_dict(orient="records"):
    out = preprocess_row(r)
    if out is not None:
        records.append(out)

with open("qwen_rca_train.jsonl", "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps({"id": rec["id"], "messages": rec["messages"]}, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} examples to qwen_rca_train.jsonl")


Saved 2400 examples to qwen_rca_train.jsonl


## 📊 Enhanced Preprocessing with Type Safety & Feature Engineering

**Key Improvements:**
1. ✅ **Type casting** - Ensures all numeric fields are properly typed (not strings)
2. ✅ **Missing neighbor RSRP fields** - Now included in float casting
3. ✅ **Centralized field definitions** - Easy to maintain
4. ✅ **RCA-friendly features** - Computed automatically for better model learning

**Why This Matters:**
- Prevents "123.4" string vs 123.4 float bugs in calculations
- Ensures distance/ratio computations work correctly
- Makes feature engineering more reliable

In [5]:
import re
import json
import math
import pandas as pd
from statistics import mean, median
from typing import List, Dict, Any, Optional

# ============================================================================
# LABEL MAPPING
# ============================================================================
CAUSE_TO_NUM = {f"C{i}": i for i in range(1, 9)}
NUM_TO_CAUSE = {i: f"C{i}" for i in range(1, 9)}

# ============================================================================
# TYPE CONVERSION UTILITIES
# ============================================================================

def to_float(x) -> Optional[float]:
    """Safely convert to float, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_int(x) -> Optional[int]:
    """Safely convert to int, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return int(float(x))  # float first to handle "123.0" strings
    except (ValueError, TypeError):
        return None

def sanitize_question_text(q: str) -> str:
    """
    Convert literal '\\n' to real newlines.
    CRITICAL: Don't use unicode decoding or you corrupt '\\boxed'
    """
    if q is None:
        return ""
    return q.replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

# ============================================================================
# DOMAIN-SPECIFIC CALCULATIONS
# ============================================================================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance in meters between two lat/lon points"""
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    """
    Map beam scenario to vertical beamwidth:
    - DEFAULT/SCENARIO_1-5: 6 degrees
    - SCENARIO_6-11: 12 degrees
    - SCENARIO_12+: 25 degrees
    """
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6
    
    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6
    
    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    elif 6 <= k <= 11:
        return 12
    else:  # 12+
        return 25

def electronic_tilt_deg(digital_tilt) -> float:
    """
    Convert digital tilt to degrees:
    - 255 => 6 degrees (special case)
    - Otherwise: value is degrees
    """
    try:
        v = int(float(digital_tilt))
        return 6.0 if v == 255 else float(v)
    except (ValueError, TypeError):
        return 6.0  # Default fallback

print("✓ Type conversion and domain utilities loaded")

✓ Type conversion and domain utilities loaded


In [6]:
# ============================================================================
# TABLE PARSING
# ============================================================================

def parse_pipe_table(table_text: str) -> List[Dict[str, Any]]:
    """
    Parse pipe-delimited table into list of dicts.
    Handles '-', '—', and empty values as None.
    """
    if not table_text:
        return []
    
    lines = [ln.strip() for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if "|" in ln]
    
    if not lines:
        return []
    
    # Parse header
    header = [h.strip() for h in lines[0].split("|")]
    
    # Parse rows
    rows = []
    for ln in lines[1:]:
        parts = [p.strip() for p in ln.split("|")]
        
        # Normalize row length to match header
        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        elif len(parts) > len(header):
            parts = parts[:len(header)]
        
        # Build record
        rec = {}
        for k, v in zip(header, parts):
            rec[k] = None if v in ("", "-", "—", "–") else v
        
        rows.append(rec)
    
    return rows

print("✓ Table parsing loaded")

✓ Table parsing loaded


In [7]:
# ============================================================================
# COLUMN MAPPING & TYPE CASTING (IMPROVED)
# ============================================================================

# Drive-test data column mappings
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",
    
    # Neighbor PCIs
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",
    
    # Neighbor RSRPs (CRITICAL - these were missing!)
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

# Engineering parameters column mappings
ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

# Define fields by type for proper casting
FLOAT_FIELDS = [
    "longitude", "latitude", "gps_speed_kmh",
    "ss_rsrp_dbm", "ss_sinr_db", "throughput_mbps", "dl_rb_num",
    "mechanical_downtilt", "digital_tilt", "height", "max_tx_power",
    "mechanical_azimuth", "digital_azimuth",
    # CRITICAL: Add neighbor RSRP fields
    "nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm", 
    "nei4_rsrp_dbm", "nei5_rsrp_dbm",
]

INT_FIELDS = [
    "pci", "serving_pci",
    "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci",
]

def normalize_rows(rows: List[Dict], mapping: Dict[str, str]) -> List[Dict]:
    """
    Normalize column names and apply type casting.
    This ensures all numeric operations work correctly.
    """
    normalized = []
    
    for r in rows:
        nr = {}
        
        # Map column names
        for k, v in r.items():
            normalized_key = mapping.get(k)
            if normalized_key is None:
                continue  # Skip unmapped columns
            nr[normalized_key] = v
        
        # Type casting - CRITICAL for numeric operations
        for field in FLOAT_FIELDS:
            if field in nr:
                nr[field] = to_float(nr[field])
        
        for field in INT_FIELDS:
            if field in nr:
                nr[field] = to_int(nr[field])
        
        normalized.append(nr)
    
    return normalized

print("✓ Column mappings and type casting configured")
print(f"  - Float fields: {len(FLOAT_FIELDS)}")
print(f"  - Int fields: {len(INT_FIELDS)}")
print(f"  - Drive-test columns mapped: {len(DRIVE_MAP)}")
print(f"  - Engineering columns mapped: {len(ENG_MAP)}")

✓ Column mappings and type casting configured
  - Float fields: 18
  - Int fields: 7
  - Drive-test columns mapped: 19
  - Engineering columns mapped: 14


In [8]:
# ============================================================================
# RCA-FRIENDLY FEATURE ENGINEERING
# ============================================================================

def compute_rca_features(drive_rows: List[Dict], eng_rows: List[Dict]) -> Dict[str, Any]:
    """
    Compute derived features that help the model learn root cause patterns.
    This is the KEY to good RCA performance!
    
    Returns dict with all computed features.
    """
    features = {}
    
    # -------------------------------------------------------------------------
    # A) Throughput-drop signature (C8 indicator)
    # -------------------------------------------------------------------------
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]
    
    if tps:
        features["tp_min_mbps"] = min(tps)
        features["tp_mean_mbps"] = mean(tps)
        features["tp_max_mbps"] = max(tps)
        features["tp_drop_ratio"] = sum(1 for x in tps if x < 600) / len(tps)
        features["tp_samples_below_600"] = sum(1 for x in tps if x < 600)
    else:
        features["tp_min_mbps"] = None
        features["tp_mean_mbps"] = None
        features["tp_max_mbps"] = None
        features["tp_drop_ratio"] = None
        features["tp_samples_below_600"] = 0
    
    # -------------------------------------------------------------------------
    # B) Resource constraint (C8)
    # -------------------------------------------------------------------------
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]
    
    if rbs:
        features["rb_mean"] = mean(rbs)
        features["rb_min"] = min(rbs)
        features["rb_max"] = max(rbs)
        features["rb_below_160_flag"] = features["rb_mean"] < 160 if features["rb_mean"] else False
    else:
        features["rb_mean"] = None
        features["rb_min"] = None
        features["rb_max"] = None
        features["rb_below_160_flag"] = False
    
    # -------------------------------------------------------------------------
    # C) Speed rule (C7)
    # -------------------------------------------------------------------------
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]
    
    if speeds:
        features["speed_max_kmh"] = max(speeds)
        features["speed_mean_kmh"] = mean(speeds)
        features["speed_above_40_flag"] = features["speed_max_kmh"] > 40 if features["speed_max_kmh"] else False
    else:
        features["speed_max_kmh"] = None
        features["speed_mean_kmh"] = None
        features["speed_above_40_flag"] = False
    
    # -------------------------------------------------------------------------
    # D) Handover / mobility (C5)
    # -------------------------------------------------------------------------
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]
    
    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1
    
    features["handover_count"] = handover_count
    features["frequent_handover_flag"] = handover_count >= 3  # Threshold for "frequent"
    
    # -------------------------------------------------------------------------
    # E) PCI mod 30 collision (C6)
    # -------------------------------------------------------------------------
    neighbor_pcis = []
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            v = r.get(k)
            if v is not None:
                neighbor_pcis.append(v)
    
    mod30_collision = False
    if serving_pcis and neighbor_pcis:
        serving_pci = serving_pcis[-1]  # Use last serving PCI
        mod30_collision = any((n % 30) == (serving_pci % 30) for n in neighbor_pcis)
    
    features["pci_mod30_collision"] = mod30_collision
    
    # -------------------------------------------------------------------------
    # F) Coverage distance / overshoot (C2)
    # -------------------------------------------------------------------------
    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}
    
    distances = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)
        
        if not cell:
            continue
        
        if (r.get("latitude") is None or r.get("longitude") is None or
            cell.get("latitude") is None or cell.get("longitude") is None):
            continue
        
        dist = haversine_m(
            r["latitude"], r["longitude"],
            cell["latitude"], cell["longitude"]
        )
        distances.append(dist)
    
    if distances:
        features["dist_median_m"] = median(distances)
        features["dist_p95_m"] = sorted(distances)[int(0.95 * (len(distances) - 1))]
        features["dist_max_m"] = max(distances)
        features["overshoot_flag"] = features["dist_p95_m"] > 1000 if features["dist_p95_m"] else False
    else:
        features["dist_median_m"] = None
        features["dist_p95_m"] = None
        features["dist_max_m"] = None
        features["overshoot_flag"] = False
    
    # -------------------------------------------------------------------------
    # G) Downtilt / weak far coverage (C1)
    # -------------------------------------------------------------------------
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])
        
        if cell:
            mech_tilt = cell.get("mechanical_downtilt") or 0.0
            elec_tilt = electronic_tilt_deg(cell.get("digital_tilt"))
            total_tilt = float(mech_tilt) + float(elec_tilt)
            
            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))
            tilt_to_beamwidth_ratio = total_tilt / vbw if vbw > 0 else 0.0
            
            features["serving_mechanical_tilt_deg"] = mech_tilt
            features["serving_electronic_tilt_deg"] = elec_tilt
            features["serving_total_tilt_deg"] = total_tilt
            features["serving_vertical_beamwidth_deg"] = vbw
            features["tilt_to_beamwidth_ratio"] = tilt_to_beamwidth_ratio
            features["large_tilt_flag"] = total_tilt > 15  # Threshold for "too large"
        else:
            features["serving_mechanical_tilt_deg"] = None
            features["serving_electronic_tilt_deg"] = None
            features["serving_total_tilt_deg"] = None
            features["serving_vertical_beamwidth_deg"] = None
            features["tilt_to_beamwidth_ratio"] = None
            features["large_tilt_flag"] = False
    else:
        features["serving_mechanical_tilt_deg"] = None
        features["serving_electronic_tilt_deg"] = None
        features["serving_total_tilt_deg"] = None
        features["serving_vertical_beamwidth_deg"] = None
        features["tilt_to_beamwidth_ratio"] = None
        features["large_tilt_flag"] = False
    
    # -------------------------------------------------------------------------
    # H) Overlapping coverage (C4)
    # -------------------------------------------------------------------------
    # Count strong neighbors (RSRP within 5dB of strongest and > -95dBm)
    neighbor_rsrps = []
    for r in drive_rows:
        for k in ["nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm", 
                  "nei4_rsrp_dbm", "nei5_rsrp_dbm"]:
            v = r.get(k)
            if v is not None:
                neighbor_rsrps.append(v)
    
    if neighbor_rsrps:
        strongest_nei = max(neighbor_rsrps)
        strong_neighbors = [
            rsrp for rsrp in neighbor_rsrps 
            if rsrp > -95 and abs(rsrp - strongest_nei) <= 5
        ]
        features["strong_neighbor_count"] = len(strong_neighbors)
        features["overlap_flag"] = len(strong_neighbors) >= 3  # Multiple strong neighbors
    else:
        features["strong_neighbor_count"] = 0
        features["overlap_flag"] = False
    
    return features

print("✓ RCA feature engineering ready")
print("  Features computed:")
print("    - Throughput metrics (tp_min, tp_mean, tp_drop_ratio)")
print("    - Resource blocks (rb_mean, rb_below_160_flag)")
print("    - Speed (speed_max, speed_above_40_flag)")
print("    - Handovers (handover_count, frequent_handover_flag)")
print("    - PCI mod 30 collisions")
print("    - Distance/overshoot (dist_median, dist_p95, overshoot_flag)")
print("    - Tilt analysis (total_tilt, tilt_to_beamwidth_ratio)")
print("    - Overlap detection (strong_neighbor_count, overlap_flag)")

✓ RCA feature engineering ready
  Features computed:
    - Throughput metrics (tp_min, tp_mean, tp_drop_ratio)
    - Resource blocks (rb_mean, rb_below_160_flag)
    - Speed (speed_max, speed_above_40_flag)
    - Handovers (handover_count, frequent_handover_flag)
    - PCI mod 30 collisions
    - Distance/overshoot (dist_median, dist_p95, overshoot_flag)
    - Tilt analysis (total_tilt, tilt_to_beamwidth_ratio)
    - Overlap detection (strong_neighbor_count, overlap_flag)


In [9]:
# ============================================================================
# PROMPT FORMATTING (Chat Template)
# ============================================================================

def format_value(v) -> str:
    """Format value for display in prompt"""
    if v is None:
        return "N/A"
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def format_user_prompt(features: Dict, drive_rows: List[Dict], eng_rows: List[Dict]) -> str:
    """
    Format the user prompt with derived features and raw data.
    This format helps the model learn RCA patterns effectively.
    """
    
    # Format feature summary
    feature_summary = "\n".join([
        "**Throughput Metrics:**",
        f"  - Min: {format_value(features.get('tp_min_mbps'))} Mbps",
        f"  - Mean: {format_value(features.get('tp_mean_mbps'))} Mbps",
        f"  - Drop ratio (<600): {format_value(features.get('tp_drop_ratio'))}",
        "",
        "**Resource Blocks:**",
        f"  - Mean RB: {format_value(features.get('rb_mean'))}",
        f"  - Min RB: {format_value(features.get('rb_min'))}",
        f"  - Below 160 threshold: {format_value(features.get('rb_below_160_flag'))}",
        "",
        "**Speed:**",
        f"  - Max speed: {format_value(features.get('speed_max_kmh'))} km/h",
        f"  - Exceeds 40 km/h: {format_value(features.get('speed_above_40_flag'))}",
        "",
        "**Handovers:**",
        f"  - Count: {format_value(features.get('handover_count'))}",
        f"  - Frequent handovers: {format_value(features.get('frequent_handover_flag'))}",
        "",
        "**PCI Collision:**",
        f"  - Mod 30 collision: {format_value(features.get('pci_mod30_collision'))}",
        "",
        "**Coverage Distance:**",
        f"  - Median: {format_value(features.get('dist_median_m'))} m",
        f"  - P95: {format_value(features.get('dist_p95_m'))} m",
        f"  - Overshoot (>1km): {format_value(features.get('overshoot_flag'))}",
        "",
        "**Tilt Analysis:**",
        f"  - Total tilt: {format_value(features.get('serving_total_tilt_deg'))}°",
        f"  - Vertical beamwidth: {format_value(features.get('serving_vertical_beamwidth_deg'))}°",
        f"  - Tilt/beamwidth ratio: {format_value(features.get('tilt_to_beamwidth_ratio'))}",
        f"  - Large tilt: {format_value(features.get('large_tilt_flag'))}",
        "",
        "**Overlap Detection:**",
        f"  - Strong neighbors: {format_value(features.get('strong_neighbor_count'))}",
        f"  - Overlap suspected: {format_value(features.get('overlap_flag'))}",
    ])
    
    # Truncate raw data to last 10 samples for brevity
    drive_sample = drive_rows[-10:] if len(drive_rows) > 10 else drive_rows
    
    prompt = f"""Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \\boxed{{}}.

**Root Cause Options:**
C1: Downtilt too large, causing weak far-end coverage
C2: Coverage distance exceeds 1km (overshoot)
C3: Neighbor cell provides higher throughput
C4: Non-colocated co-frequency neighbors cause severe overlapping coverage
C5: Frequent handovers degrade performance
C6: Neighbor and serving cell have same PCI mod 30 (interference)
C7: Vehicle speed exceeds 40 km/h
C8: Average scheduled RBs below 160

**Given Rules:**
- Digital tilt value 255 = 6 degrees; otherwise value represents actual degrees
- Beam scenario vertical beamwidth mapping:
  * DEFAULT or SCENARIO_1-5: 6 degrees
  * SCENARIO_6-11: 12 degrees
  * SCENARIO_12+: 25 degrees

**Derived Feature Summary:**
{feature_summary}

**Drive-Test Data (last {len(drive_sample)} samples):**
```json
{json.dumps(drive_sample, indent=2, ensure_ascii=False)}
```

**Engineering Parameters:**
```json
{json.dumps(eng_rows, indent=2, ensure_ascii=False)}
```"""
    
    return prompt

print("✓ Prompt formatting ready")

✓ Prompt formatting ready


In [10]:
# ============================================================================
# MAIN PREPROCESSING FUNCTION
# ============================================================================

def preprocess_row(row: Dict) -> Optional[Dict]:
    """
    Main preprocessing pipeline:
    1. Sanitize text
    2. Parse tables
    3. Normalize columns with type casting
    4. Compute RCA features
    5. Format prompt
    6. Return chat messages
    
    Returns None if row cannot be processed.
    """
    # Sanitize question text
    question_text = sanitize_question_text(row["question"])
    label = str(row["answer"]).strip().upper()
    
    # Validate label
    if label not in CAUSE_TO_NUM:
        return None
    
    # Extract table sections
    drive_match = re.search(r"User plane drive test data as follows[:：]\s*", question_text)
    eng_match = re.search(r"Engeneering parameters data as follows[:：]\s*", question_text)
    
    if not drive_match or not eng_match or eng_match.start() <= drive_match.end():
        return None
    
    drive_text = question_text[drive_match.end():eng_match.start()].strip()
    eng_text = question_text[eng_match.end():].strip()
    
    # Parse tables
    drive_raw = parse_pipe_table(drive_text)
    eng_raw = parse_pipe_table(eng_text)
    
    if not drive_raw or not eng_raw:
        return None
    
    # Normalize with type casting (CRITICAL!)
    drive_rows = normalize_rows(drive_raw, DRIVE_MAP)
    eng_rows = normalize_rows(eng_raw, ENG_MAP)
    
    # Compute RCA features
    features = compute_rca_features(drive_rows, eng_rows)
    
    # Format user prompt
    user_content = format_user_prompt(features, drive_rows, eng_rows)
    
    # Convert label to number
    answer_num = CAUSE_TO_NUM[label]
    
    # Assistant response (short and focused)
    assistant_content = f"Based on the analysis of the derived features and network data, the most likely root cause is {label}.\n\nFinal answer: \\\\boxed{{{answer_num}}}"
    
    # Return in chat format
    return {
        "id": row["ID"],
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a senior 5G RAN optimization engineer. "
                    "Analyze the provided drive-test KPIs, derived features, and engineering parameters. "
                    "Identify the most likely root cause (C1–C8) for throughput degradation. "
                    "Provide brief reasoning and end with \\\\boxed{n}."
                )
            },
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
        ],
        # Also save features for analysis
        "features": features
    }

print("✓ Main preprocessing pipeline ready")

✓ Main preprocessing pipeline ready


## 🚀 Apply Enhanced Preprocessing

Now let's process the training data with the improved pipeline that includes:
- Proper type casting (no more string/number bugs)
- RCA-friendly feature engineering
- Enhanced chat formatting

In [11]:
# Process all training samples
print("=" * 70)
print("PROCESSING TRAINING DATA WITH ENHANCED PIPELINE")
print("=" * 70)
print(f"Total samples to process: {len(df_train):,}\n")

processed_records = []
failed_count = 0

for idx, row_dict in enumerate(df_train.to_dict(orient="records")):
    if idx % 10000 == 0 and idx > 0:
        print(f"  Processed {idx:,} samples... ({len(processed_records):,} successful)")
    
    result = preprocess_row(row_dict)
    
    if result is not None:
        processed_records.append(result)
    else:
        failed_count += 1

print(f"\n{'='*70}")
print(f"✓ Processing complete!")
print(f"  - Successfully processed: {len(processed_records):,}")
print(f"  - Failed: {failed_count:,}")
print(f"  - Success rate: {100 * len(processed_records) / len(df_train):.1f}%")
print(f"{'='*70}\n")

PROCESSING TRAINING DATA WITH ENHANCED PIPELINE
Total samples to process: 2,400


✓ Processing complete!
  - Successfully processed: 2,400
  - Failed: 0
  - Success rate: 100.0%


✓ Processing complete!
  - Successfully processed: 2,400
  - Failed: 0
  - Success rate: 100.0%



In [12]:
# Analyze the computed features
print("FEATURE ANALYSIS")
print("=" * 70)

# Extract all features for analysis
all_features = [rec["features"] for rec in processed_records]

# Analyze key RCA indicators
feature_stats = {}
for key in all_features[0].keys():
    values = [f.get(key) for f in all_features if f.get(key) is not None]
    
    if len(values) > 0:
        if isinstance(values[0], bool):
            feature_stats[key] = {
                "type": "boolean",
                "true_count": sum(values),
                "false_count": len(values) - sum(values),
                "true_pct": 100 * sum(values) / len(values)
            }
        elif isinstance(values[0], (int, float)):
            feature_stats[key] = {
                "type": "numeric",
                "min": min(values),
                "max": max(values),
                "mean": sum(values) / len(values),
                "count": len(values)
            }

# Display key statistics
print("\n🎯 RCA INDICATOR STATISTICS:\n")

print("**Throughput Issues (C8 related):**")
if "tp_drop_ratio" in feature_stats:
    s = feature_stats["tp_drop_ratio"]
    print(f"  Drop ratio: mean={s['mean']:.2f}, min={s['min']:.2f}, max={s['max']:.2f}")
if "rb_below_160_flag" in feature_stats:
    s = feature_stats["rb_below_160_flag"]
    print(f"  RB below 160: {s['true_pct']:.1f}% of samples")

print("\n**Speed Issues (C7):**")
if "speed_above_40_flag" in feature_stats:
    s = feature_stats["speed_above_40_flag"]
    print(f"  Speed > 40 km/h: {s['true_pct']:.1f}% of samples")

print("\n**Handover Issues (C5):**")
if "frequent_handover_flag" in feature_stats:
    s = feature_stats["frequent_handover_flag"]
    print(f"  Frequent handovers: {s['true_pct']:.1f}% of samples")

print("\n**PCI Collision (C6):**")
if "pci_mod30_collision" in feature_stats:
    s = feature_stats["pci_mod30_collision"]
    print(f"  PCI mod 30 collision: {s['true_pct']:.1f}% of samples")

print("\n**Overshoot (C2):**")
if "overshoot_flag" in feature_stats:
    s = feature_stats["overshoot_flag"]
    print(f"  Overshoot (>1km): {s['true_pct']:.1f}% of samples")

print("\n**Tilt Issues (C1):**")
if "large_tilt_flag" in feature_stats:
    s = feature_stats["large_tilt_flag"]
    print(f"  Large tilt (>15°): {s['true_pct']:.1f}% of samples")

print("\n**Overlap (C4):**")
if "overlap_flag" in feature_stats:
    s = feature_stats["overlap_flag"]
    print(f"  Overlap detected: {s['true_pct']:.1f}% of samples")

print(f"\n{'='*70}")

FEATURE ANALYSIS

🎯 RCA INDICATOR STATISTICS:

**Throughput Issues (C8 related):**
  Drop ratio: mean=0.39, min=0.00, max=0.40
  RB below 160: 1.7% of samples

**Speed Issues (C7):**
  Speed > 40 km/h: 14.5% of samples

**Handover Issues (C5):**
  Frequent handovers: 14.7% of samples

**PCI Collision (C6):**
  PCI mod 30 collision: 100.0% of samples

**Overshoot (C2):**
  Overshoot (>1km): 13.3% of samples

**Tilt Issues (C1):**
  Large tilt (>15°): 36.0% of samples

**Overlap (C4):**
  Overlap detected: 68.5% of samples



In [13]:
# Show an example formatted prompt
print("EXAMPLE FORMATTED PROMPT")
print("=" * 70)
print("\nHere's what the model will see during training:\n")

example = processed_records[0]
print(f"ID: {example['id']}")
print(f"\nSystem Message:\n{example['messages'][0]['content']}")
print(f"\n{'─'*70}")
print(f"\nUser Message (first 1500 chars):\n{example['messages'][1]['content'][:1500]}...")
print(f"\n{'─'*70}")
print(f"\nAssistant Response:\n{example['messages'][2]['content']}")
print(f"\n{'='*70}")

EXAMPLE FORMATTED PROMPT

Here's what the model will see during training:

ID: ID_1P7PJMPV0R

System Message:
You are a senior 5G RAN optimization engineer. Analyze the provided drive-test KPIs, derived features, and engineering parameters. Identify the most likely root cause (C1–C8) for throughput degradation. Provide brief reasoning and end with \\boxed{n}.

──────────────────────────────────────────────────────────────────────

User Message (first 1500 chars):
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{}.

**Root Cause Options:**
C1: Downtilt too large, causing weak far-end coverage
C2: Coverage distance exceeds 1km (overshoot)
C3: Neighbor cell provides higher throughput
C4: Non-colocated co-frequency neighbors cause severe overlapping coverage
C5: F

## 💾 Save Enhanced Training Data

Save in two formats:
1. **JSONL** - For external training (QWEN fine-tuning APIs, etc.)
2. **HuggingFace Dataset** - For local training with Transformers

In [14]:
# Save as JSONL (industry standard format)
jsonl_path = "qwen_rca_train_enhanced.jsonl"

print(f"Saving to {jsonl_path}...")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for rec in processed_records:
        # Save only ID and messages (not features)
        output_rec = {
            "id": rec["id"],
            "messages": rec["messages"]
        }
        f.write(json.dumps(output_rec, ensure_ascii=False) + "\n")

print(f"✓ Saved {len(processed_records):,} examples to {jsonl_path}")
print(f"  File size: {Path(jsonl_path).stat().st_size / 1024 / 1024:.1f} MB\n")

# Also save features separately for analysis
features_path = "qwen_rca_features.jsonl"
print(f"Saving features to {features_path}...")
with open(features_path, "w", encoding="utf-8") as f:
    for rec in processed_records:
        feature_rec = {
            "id": rec["id"],
            "features": rec["features"]
        }
        f.write(json.dumps(feature_rec, ensure_ascii=False) + "\n")

print(f"✓ Saved features to {features_path}")
print(f"  File size: {Path(features_path).stat().st_size / 1024 / 1024:.1f} MB")

Saving to qwen_rca_train_enhanced.jsonl...
✓ Saved 2,400 examples to qwen_rca_train_enhanced.jsonl
  File size: 25.6 MB

Saving features to qwen_rca_features.jsonl...
✓ Saved features to qwen_rca_features.jsonl
  File size: 1.8 MB


In [15]:
# Prepare data for HuggingFace Trainer
from datasets import Dataset

# Format for training: apply chat template
def format_for_training(example):
    """Format messages into training text"""
    # For now, return the messages as-is - tokenizer will handle chat template
    return {
        "messages": example["messages"]
    }

# Create dataset
train_data = [{"messages": rec["messages"]} for rec in processed_records]
hf_dataset = Dataset.from_list(train_data)

print(f"\n✓ Created HuggingFace dataset: {len(hf_dataset):,} samples")
print(f"  Columns: {hf_dataset.column_names}")
print(f"  Features: {hf_dataset.features}")

# Save to disk for quick loading later
hf_dataset.save_to_disk("qwen_rca_dataset_enhanced")
print(f"✓ Saved HuggingFace dataset to 'qwen_rca_dataset_enhanced/'")

# Create a small validation split
train_test = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test["train"]
val_dataset = train_test["test"]

print(f"\n📊 Dataset split:")
print(f"  Training: {len(train_dataset):,} samples")
print(f"  Validation: {len(val_dataset):,} samples")


✓ Created HuggingFace dataset: 2,400 samples
  Columns: ['messages']
  Features: {'messages': List({'content': Value('string'), 'role': Value('string')})}


Saving the dataset (1/1 shards): 100%|██████████| 2400/2400 [00:00<00:00, 238121.06 examples/s]

✓ Saved HuggingFace dataset to 'qwen_rca_dataset_enhanced/'

📊 Dataset split:
  Training: 2,160 samples
  Validation: 240 samples


In [16]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 2160
})

In [17]:
train_dataset['messages'][2]

[{'content': 'You are a senior 5G RAN optimization engineer. Analyze the provided drive-test KPIs, derived features, and engineering parameters. Identify the most likely root cause (C1–C8) for throughput degradation. Provide brief reasoning and end with \\\\boxed{n}.',
  'role': 'system'},
 {'content': 'Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following 8 potential root causes, select the most likely one and enclose its number in \\boxed{}.\n\n**Root Cause Options:**\nC1: Downtilt too large, causing weak far-end coverage\nC2: Coverage distance exceeds 1km (overshoot)\nC3: Neighbor cell provides higher throughput\nC4: Non-colocated co-frequency neighbors cause severe overlapping coverage\nC5: Frequent handovers degrade performance\nC6: Neighbor and serving cell have same PCI mod 30 (interference)\nC7: Vehicle speed exceeds 40 km/h\nC8: Average 

In [18]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [19]:
# Step 2: Imports
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset
import torch

In [20]:
# Step 4: Load Tokenizer and Model (optimized for GPU with quantization)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Set pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {model_name}")

# Configure 4-bit quantization for GPU memory efficiency
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    

    model = AutoModelForCausalLM.from_pretrained(    print(f"✓ GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

        model_name,if torch.cuda.is_available():

        quantization_config=bnb_config,# Print model memory footprint

        device_map="auto",

        trust_remote_code=True,print(f"✓ Tokenizer pad token: {tokenizer.pad_token}")

    )print(f"✓ Model: {model_name}")

    

    # Prepare model for k-bit training    print(f"✓ Model loaded without quantization (CPU mode)")

    model = prepare_model_for_kbit_training(model)    )

    print(f"✓ Model loaded with 4-bit quantization")        low_cpu_mem_usage=True,

else:        torch_dtype=torch.float32,

    # Fallback for CPU (not recommended for training)        model_name,
    model = AutoModelForCausalLM.from_pretrained(

`torch_dtype` is deprecated! Use `dtype` instead!


✓ Model loaded: Qwen/Qwen2.5-1.5B-Instruct
✓ Model dtype: torch.float32
✓ Device: Will be moved by Trainer


In [21]:
# Step 5: Apply LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=16,  # Increased rank for better capacity
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target all attention matrices
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()    print(f"✓ GPU memory after LoRA: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

if torch.cuda.is_available():
print("\n✓ LoRA applied successfully")

In [22]:
# Step 6: Tokenize with Chat Template
def tokenize_chat(examples):
    """
    Tokenize chat messages using the model's chat template.
    Handles batched processing correctly.
    """
    # Apply chat template to each example's messages
    texts = []
    for messages in examples["messages"]:
        # Apply chat template (this formats system/user/assistant properly)
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False  # Don't add prompt since we have assistant response
        )
        texts.append(text)
    
    # Tokenize all texts at once (batched)
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,  # Will be handled by data collator
        max_length=2048,  # Reduced from 6000 for memory efficiency
        return_tensors=None  # Return lists, not tensors (for datasets.map)
    )
    
    # Copy input_ids to labels for causal LM training
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

print("Tokenizing datasets with chat template...")
train_dataset = train_dataset.map(
    tokenize_chat, 
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)
val_dataset = val_dataset.map(
    tokenize_chat, 
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print(f"✓ Tokenized {len(train_dataset)} training samples")
print(f"✓ Tokenized {len(val_dataset)} validation samples")
print(f"Sample tokenized output keys: {train_dataset.column_names}")

Tokenizing datasets with chat template...


Tokenizing validation: 100%|██████████| 240/240 [00:00<00:00, 552.00 examples/s]

✓ Tokenized 2160 training samples
✓ Tokenized 240 validation samples
Sample tokenized output keys: ['input_ids', 'attention_mask', 'labels']


In [23]:
from transformers import DataCollatorForLanguageModeling, Trainer

# Data collator for dynamic padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

# Training arguments optimized for GPU
output_dir = "./qwen_telco_finetuned"

# Adjust batch size based on GPU availability
if torch.cuda.is_available():
    # GPU settings - adjust based on your GPU memory
    per_device_batch = 2  # Start with 2, increase if you have more VRAM
    gradient_accum = 4    # Effective batch size = 2 * 4 = 8
    use_fp16 = torch.cuda.is_bf16_supported() == False  # Use FP16 if BF16 not supported
    use_bf16 = torch.cuda.is_bf16_supported()  # Use BF16 if supported (better for training)
    optim_choice = "paged_adamw_8bit"  # Memory-efficient optimizer for GPU
else:
    # CPU fallback (not recommended)
    per_device_batch = 1
    gradient_accum = 8
    use_fp16 = False
    use_bf16 = False
    optim_choice = "adamw_torch"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,  # Increased for better convergence
    per_device_train_batch_size=per_device_batch,
    per_device_eval_batch_size=per_device_batch,
    gradient_accumulation_steps=gradient_accum,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=use_fp16,
    bf16=use_bf16,

    optim=optim_choice,print("=" * 70)

    gradient_checkpointing=True,  # Save memoryprint(f"  Total steps: ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

    max_grad_norm=1.0,print(f"  Optimizer: {training_args.optim}")

    lr_scheduler_type="cosine",print(f"  Learning rate: {training_args.learning_rate}")

    report_to="none",  # Change to "tensorboard" if you want loggingprint(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

    remove_unused_columns=False,print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

    dataloader_num_workers=2,  # Parallel data loadingprint(f"  Batch size per device: {training_args.per_device_train_batch_size}")

    group_by_length=True,  # Group similar lengths for efficiencyprint(f"  Epochs: {training_args.num_train_epochs}")

)    print(f"  Precision: {'BF16' if use_bf16 else 'FP16' if use_fp16 else 'FP32'}")

    print(f"  GPU: {torch.cuda.get_device_name(0)}")

print("=" * 70)if torch.cuda.is_available():

print("TRAINING CONFIGURATION")print(f"  Device: {device}")
print("=" * 70)

Training configuration:
  Epochs: 1
  Batch size per device: 1
  Gradient accumulation steps: 8
  Effective batch size: 8
  Learning rate: 0.0002
  Total training steps: ~270


In [24]:
# Clear GPU cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

print("=" * 70 + "\n")    raise

        print(f"GPU memory at failure: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Start training with error handling    if torch.cuda.is_available():

try:    print(f"\n❌ Training failed with error: {e}")

    trainer.train()except Exception as e:

    print("\n" + "=" * 70)        

    print("✓ TRAINING COMPLETED SUCCESSFULLY!")        print(f"✓ Final GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

    print("=" * 70)    if torch.cuda.is_available():

        

    # Save the final model    print(f"\n✓ Model saved to: {output_dir}")
    trainer.save_model(output_dir)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# Training complete! Model is ready to use.
# Check the cells above for:
# - Saving the model
# - Testing inference
# - Downloading to your local machine

print("✓ Notebook execution complete!")
print("\nNext steps:")
print("  1. Save your trained model (see cell above)")
print("  2. Test inference with validation samples")
print("  3. Download model or save to Google Drive")
print("  4. Use the model for predictions on test set")

## 💾 Save and Download Trained Model

After training completes, you can save and download your model.

In [ ]:
# Save the final trained model and tokenizer
final_output_dir = "./qwen_telco_final"

print("Saving final model and tokenizer...")
model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

print(f"✓ Model saved to: {final_output_dir}")
print("\nTo download the model from Colab:")
print("  1. Go to Files panel (left sidebar)")
print(f"  2. Navigate to {final_output_dir}")
print("  3. Right-click and select 'Download'")
print("\nOr use Google Drive to save it:")
print("  from google.colab import drive")
print("  drive.mount('/content/drive')")
print("  !cp -r ./qwen_telco_final /content/drive/MyDrive/")

## 🧪 Test Inference (Optional)

Test your trained model with a real sample from the validation set. This cell will:
1. Load an actual validation sample with all the network data
2. Generate a prediction using the fine-tuned model
3. Compare predicted vs expected root cause

**Note:** This requires that you've run all the preprocessing cells above to have `processed_records` available.

In [ ]:
# Test the trained model with actual data from validation set
import re

def extract_boxed_answer(text):
    """Extract the number from \\boxed{n} format"""
    match = re.search(r'\\boxed\{(\d)\}', text)
    if match:
        return int(match.group(1))
    return None

# We need to get the original messages before tokenization
# Let's reload one sample from the processed records
print("Loading a test sample from processed records...")

# Get one example from validation set (using the original processed_records)
# Since we split 90/10, let's use an index that would be in validation
import random
random.seed(42)
val_indices = random.sample(range(len(processed_records)), k=int(0.1 * len(processed_records)))

# Get first validation sample
test_record = processed_records[val_indices[0]]
test_messages = test_record["messages"]

print(f"\nTest sample ID: {test_record['id']}")
print(f"Number of messages: {len(test_messages)}")
print(f"\nSystem message length: {len(test_messages[0]['content'])} chars")
print(f"User message length: {len(test_messages[1]['content'])} chars")
print(f"\nExpected answer: {test_messages[2]['content']}")

# Apply chat template for inference (without assistant response, add generation prompt)
inference_messages = test_messages[:2]  # Only system and user messages

test_text = tokenizer.apply_chat_template(
    inference_messages, 
    tokenize=False, 
    add_generation_prompt=True
)
test_inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=2048)

# Move to device
if torch.cuda.is_available():
    test_inputs = {k: v.to("cuda") for k, v in test_inputs.items()}

# Generate
print("\n" + "="*70)
print("GENERATING PREDICTION...")
print("="*70)

with torch.no_grad():
    outputs = model.generate(
        **test_inputs,
        max_new_tokens=200,
        temperature=0.3,  # Lower temperature for more focused answers
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode response (only the generated part)
generated_ids = outputs[0][len(test_inputs["input_ids"][0]):]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n" + "="*70)
print("MODEL RESPONSE:")
print("="*70)
print(response)
print("="*70)

# Extract answer
predicted = extract_boxed_answer(response)
expected_match = re.search(r'\\boxed\{(\d)\}', test_messages[2]['content'])
expected = int(expected_match.group(1)) if expected_match else None

print("\n" + "="*70)
print("RESULTS:")
print("="*70)
if predicted:
    print(f"✓ Predicted root cause: C{predicted}")
else:
    print("⚠ Could not extract \\boxed{} answer from response")

if expected:
    print(f"✓ Expected root cause: C{expected}")
    if predicted == expected:
        print("🎉 PREDICTION MATCHES EXPECTED ANSWER!")
    else:
        print("❌ Prediction does not match expected answer")
        
print("="*70)